# 02 — SigLIP2 Stage C (LoRA) Training

Runs Stage C (LoRA adapters, r=8, on q_proj/v_proj across all 12 layers of both towers) via `configs/exp03_stage_c_lora.yaml`, and plots the resulting training/validation curves.

**Why LoRA (Stage C), after Stage A and Stage B:**
- Stage A (frozen backbone, ~400K trainable params): ROC-AUC 0.651, PR-AUC 0.167 -- still behind the `text_only_bert` baseline (0.693 / 0.211).
- Stage B (unfreeze top 4 layers, 57.1M trainable params): overfit almost immediately -- val PR-AUC peaked at epoch 1, early-stopped at epoch 4, did not beat Stage A.
- Stage C (LoRA, ~986K trainable params -- more than Stage A's head alone, far less than Stage B's full unfreeze): targets the middle ground.

**Note on Stage A/B curves:** step-level loss logging (`train_log.csv` / `val_log.csv` per run) was only added once we needed a presentable artifact for Stage C -- Stage A and B's per-step curves were only ever printed to a terminal and were not preserved. Their final epoch metrics are recorded above from the experiment log; only Stage C has a full reproducible curve.

In [ ]:
import sys
sys.path.append("..")   # so `from src...` imports work when running from notebooks/

import pandas as pd
import matplotlib.pyplot as plt


## 1. Run Stage C training

This runs the actual training script as a subprocess (`!`) so every line of real output is captured in this cell and saved with the notebook. It also writes `experiments/siglip2_stage_c_lora/train_log.csv` and `val_log.csv` incrementally, so progress survives even if this cell/kernel is interrupted.

**This takes a while** (LoRA on the full ~87K-row train set, up to 10 epochs with early stopping) -- expect this cell to stay busy for a few hours.

In [ ]:
!cd .. && python3 -u -m src.training.train --config configs/exp03_stage_c_lora.yaml


## 2. Load the logged curves and plot

In [ ]:
run_dir = "../experiments/siglip2_stage_c_lora"

train_log = pd.read_csv(f"{run_dir}/train_log.csv")
val_log = pd.read_csv(f"{run_dir}/val_log.csv")

print("Training steps logged:", len(train_log))
print("Validation epochs logged:", len(val_log))
val_log


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_log["step"], train_log["loss"])
axes[0].set_xlabel("step")
axes[0].set_ylabel("train loss")
axes[0].set_title("Stage C training loss")

axes[1].plot(val_log["epoch"], val_log["pr_auc"], marker="o", label="PR-AUC")
axes[1].plot(val_log["epoch"], val_log["roc_auc"], marker="o", label="ROC-AUC")
axes[1].axhline(0.211, color="gray", linestyle="--", label="text_only_bert PR-AUC (0.211)")
axes[1].set_xlabel("epoch")
axes[1].set_title("Stage C validation metrics")
axes[1].legend()

plt.tight_layout()
plt.show()


## 3. Compare against every other model tried

Final numbers as recorded in the experiment log (`project_explanation_HE.md` section 11) -- Stage C's row is read live from `val_log.csv` above rather than hardcoded.

In [ ]:
best_stage_c = val_log.loc[val_log["pr_auc"].idxmax()]

results = pd.DataFrame([
    {"model": "tfidf_logreg",        "roc_auc": 0.660, "pr_auc": 0.208, "f1": 0.245},
    {"model": "text_only_bert",       "roc_auc": 0.693, "pr_auc": 0.211, "f1": 0.255},
    {"model": "image_only",           "roc_auc": 0.619, "pr_auc": 0.148, "f1": 0.229},
    {"model": "title_image_frozen",   "roc_auc": 0.619, "pr_auc": 0.147, "f1": 0.214},
    {"model": "siglip2_stage_a",      "roc_auc": 0.651, "pr_auc": 0.167, "f1": None},
    {"model": "siglip2_stage_b",      "roc_auc": None,  "pr_auc": None,  "f1": None},  # did not beat Stage A; exact figure only in terminal history
    {"model": "siglip2_stage_c_lora", "roc_auc": best_stage_c["roc_auc"], "pr_auc": best_stage_c["pr_auc"], "f1": best_stage_c["f1"]},
]).set_index("model")

results.sort_values("pr_auc", ascending=False)
